# Module 7: Multiple Regression

## Module Introduction

Welcome to Module 7 of Statistical Inference and Modeling!

In the previous module, we studied **simple linear regression**, where a single predictor variable was used to explain or predict an outcome. But in many real-world scenarios, outcomes are influenced by **several variables**.

For example, **house prices** may depend on:
- Size (square feet)
- Number of bedrooms
- Age of the house
- Location rating
- And many other factors!

### Learning Objectives:
1. **Construct** a multiple linear regression model with several predictors
2. **Evaluate** the overall model fit using R-squared and Adjusted R-squared
3. **Interpret** coefficients in the context of multiple predictors
4. **Test for significance** using F-tests and t-tests
5. **Check for multicollinearity** using Variance Inflation Factor (VIF)

Let's get started! 🏠📊

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Set random seed for reproducibility
np.random.seed(42)

print("✅ Setup Complete! All libraries imported successfully.")

## Section 1: Quick Review - From Simple to Multiple Regression

Before diving into multiple regression, let's quickly review **simple linear regression** where we use **one predictor** to explain an outcome:

**Simple Linear Regression:** $Y = \beta_0 + \beta_1 X + \epsilon$

But the real world is rarely that simple! Usually, an outcome is influenced by many different factors working together.

In [ ]:
# Simple Linear Regression Example: Price vs Size only
n_samples = 100

# Generate data for simple regression
size_simple = np.random.normal(2000, 500, n_samples)
price_simple = 50 + 0.15 * size_simple + np.random.normal(0, 30, n_samples)

# Visualize simple linear regression
plt.figure(figsize=(10, 5))
plt.scatter(size_simple, price_simple, alpha=0.6, s=50)
plt.plot(size_simple, 50 + 0.15 * size_simple, 'r-', linewidth=2, label='Regression Line')
plt.xlabel('Size (Square Feet)', fontsize=12)
plt.ylabel('Price ($1000s)', fontsize=12)
plt.title('Simple Linear Regression: House Price vs Size Only', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("In simple regression, we model the relationship between TWO variables.")
print("But what about other important factors like bedrooms, age, location?")

## Section 2: The Multiple Regression Model

### Moving Beyond One Variable

In multiple linear regression, we extend our model to include **multiple predictor variables**:

**Multiple Linear Regression:**
$$Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + ... + \beta_k X_k + \epsilon$$

Where:
- $Y$ = Response variable (e.g., House Price)
- $X_1, X_2, ..., X_k$ = Predictor variables (Size, Bedrooms, Age, etc.)
- $\beta_0$ = Intercept term
- $\beta_1, \beta_2, ..., \beta_k$ = Coefficients for each predictor
- $\epsilon$ = Error term

### The Hyperplane Concept
In simple regression, we fit a **line**. In multiple regression with:
- 2 predictors → We fit a **plane** in 3D space
- 3+ predictors → We fit a **hyperplane** in higher dimensions

In [ ]:
# Generate comprehensive housing dataset aligned with Week7.docx examples
np.random.seed(42)
n_houses = 200

# Generate predictor variables
size = np.random.normal(2000, 500, n_houses)  # Square feet
bedrooms = np.random.choice([1, 2, 3, 4, 5], n_houses, p=[0.1, 0.25, 0.35, 0.25, 0.05])
age = np.random.uniform(0, 50, n_houses)  # Age in years

# Location rating (1-10 scale, higher is better)
# Cities mentioned in Week7.docx: Delhi, Bangalore (Tier 1) vs Tier 2 cities
location_rating = np.random.choice(
    [3, 4, 5, 6, 7, 8, 9, 10],  # Lower ratings for Tier 2, higher for Tier 1
    n_houses,
    p=[0.05, 0.10, 0.15, 0.20, 0.20, 0.15, 0.10, 0.05]
)

# Generate house prices with realistic relationships
# Price = base + size_effect + bedroom_effect + age_effect + location_effect + noise
price = (
    30 +  # Base price
    0.12 * size +  # $120 per square foot
    15 * bedrooms +  # $15k per bedroom
    -1.5 * age +  # Depreciation: -$1.5k per year
    8 * location_rating +  # Location premium
    np.random.normal(0, 25, n_houses)  # Random noise
)

# Create DataFrame
housing_data = pd.DataFrame({
    'Price': price,
    'Size': size,
    'Bedrooms': bedrooms,
    'Age': age,
    'Location_Rating': location_rating
})

print("🏠 Housing Dataset Created!")
print(f"Number of houses: {n_houses}")
print("\nFirst few rows of our data:")
print(housing_data.head(10).round(2))
print("\n" + "="*60)
print("Dataset Statistics:")
print(housing_data.describe().round(2))

### 📝 TO DO #1: Explore the Data

**Try:**
1. Run the cell below and observe the distribution of each variable
2. Try changing the `n_houses` parameter in the cell above from 200 to 500
3. Notice how the distributions become smoother with more data
4. Question: Which variable shows the most variation? Why might this be important for prediction?

In [ ]:
# Visualize distributions of all variables
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Distribution of Housing Variables', fontsize=16, fontweight='bold')

# Price distribution
axes[0, 0].hist(housing_data['Price'], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Price ($1000s)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('House Prices')

# Size distribution
axes[0, 1].hist(housing_data['Size'], bins=30, color='green', edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Size (sq ft)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('House Size')

# Bedrooms distribution
axes[0, 2].hist(housing_data['Bedrooms'], bins=5, color='orange', edgecolor='black', alpha=0.7)
axes[0, 2].set_xlabel('Number of Bedrooms')
axes[0, 2].set_ylabel('Frequency')
axes[0, 2].set_title('Bedrooms')

# Age distribution
axes[1, 0].hist(housing_data['Age'], bins=30, color='red', edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Age (years)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('House Age')

# Location Rating distribution
axes[1, 1].hist(housing_data['Location_Rating'], bins=8, color='purple', edgecolor='black', alpha=0.7)
axes[1, 1].set_xlabel('Location Rating (1-10)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Location Rating')

# Hide the last subplot
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

### Understanding Relationships Between Variables

Before building our model, let's examine how our variables relate to each other and to the price.

In [ ]:
# Calculate correlation matrix
correlation_matrix = housing_data.corr()

# Create correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0,
            fmt='.3f', square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix of Housing Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Identify strongest correlations with Price
price_correlations = correlation_matrix['Price'].drop('Price').sort_values(ascending=False)
print("\n📊 Correlation with Price (sorted by strength):")
print("="*40)
for var, corr in price_correlations.items():
    strength = "Strong" if abs(corr) > 0.5 else "Moderate" if abs(corr) > 0.3 else "Weak"
    direction = "Positive" if corr > 0 else "Negative"
    print(f"{var:15} | {corr:+.3f} | {strength} {direction}")

## Section 3: Visualizing the Regression Hyperplane

As mentioned in the lecture, with multiple predictors we're no longer fitting a line but a **hyperplane**.
Let's visualize this in 3D with two predictors first.

In [ ]:
# Create 3D visualization with Size and Bedrooms as predictors
fig = px.scatter_3d(housing_data,
                    x='Size',
                    y='Bedrooms',
                    z='Price',
                    color='Price',
                    color_continuous_scale='viridis',
                    title='3D Visualization: Price vs Size and Bedrooms',
                    labels={'Price': 'Price ($1000s)',
                           'Size': 'Size (sq ft)',
                           'Bedrooms': 'Bedrooms'},
                    opacity=0.7)

# Update layout for better interactivity
fig.update_layout(
    scene=dict(
        xaxis_title='Size (sq ft)',
        yaxis_title='Number of Bedrooms',
        zaxis_title='Price ($1000s)',
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.5))
    ),
    height=600,
    showlegend=True
)

# Add annotation
fig.add_annotation(
    text="🔄 Drag to rotate | Scroll to zoom | Double-click to reset",
    xref="paper", yref="paper",
    x=0.5, y=-0.05, showarrow=False,
    font=dict(size=12)
)

fig.show()

print("💡 This is your data in 3D space!")
print("Notice how the points form a pattern - this is what we'll fit with our regression plane.")

## Section 4: The Least Squares Method in Multiple Regression

Just like in simple regression, we use the **Least Squares Method** to find the best-fitting hyperplane.

### The Goal:
Minimize the **Sum of Squared Errors (SSE)**:
$$SSE = \sum_{i=1}^{n} (Y_i - \hat{Y}_i)^2$$

Where:
- $Y_i$ = Actual price of house $i$
- $\hat{Y}_i$ = Predicted price from our model

The least squares method finds the coefficients ($\beta_0, \beta_1, ..., \beta_k$) that minimize this SSE.

In [ ]:
# Prepare data for multiple regression
# X contains all predictor variables (independent variables)
X = housing_data[['Size', 'Bedrooms', 'Age', 'Location_Rating']]
# y contains the target variable (dependent variable) we want to predict
y = housing_data['Price']

# Add constant (intercept) to the model
# This adds a column of 1s to X, allowing the model to fit an intercept term (β₀)
# Without this, regression line would be forced through origin (0,0)
X_with_constant = sm.add_constant(X)

# Fit the multiple regression model using Ordinary Least Squares (OLS)
# OLS finds coefficients that minimize the sum of squared residuals
# .fit() actually performs the regression calculation
model = sm.OLS(y, X_with_constant).fit()

# Display the regression results
# This shows coefficients, p-values, R-squared, and diagnostic statistics
print("="*70)
print("MULTIPLE REGRESSION RESULTS")
print("="*70)
print(model.summary())

### Interpreting the Coefficients

Each coefficient ($\beta$) represents the **estimated change in price** for a **one-unit increase** in that predictor, **holding all other variables constant**.

This is the key difference from simple regression - we're measuring the effect of each variable **while controlling for the others**.

In [ ]:
# Extract and display coefficients
coefficients = model.params
std_errors = model.bse

print("🏠 COEFFICIENT INTERPRETATION")
print("="*70)
print(f"Intercept (β₀): ${coefficients['const']:.2f}k")
print("  → Base price when all predictors = 0 (not practically meaningful)\n")

for var in ['Size', 'Bedrooms', 'Age', 'Location_Rating']:
    coef = coefficients[var]
    se = std_errors[var]

    if var == 'Size':
        print(f"Size (β₁): {coef:.4f} ± {se:.4f}")
        print(f"  → Each additional square foot increases price by ${coef*1000:.2f}")
    elif var == 'Bedrooms':
        print(f"\nBedrooms (β₂): {coef:.2f} ± {se:.2f}")
        print(f"  → Each additional bedroom increases price by ${coef:.2f}k")
    elif var == 'Age':
        print(f"\nAge (β₃): {coef:.2f} ± {se:.2f}")
        print(f"  → Each additional year of age decreases price by ${abs(coef):.2f}k")
    else:  # Location_Rating
        print(f"\nLocation Rating (β₄): {coef:.2f} ± {se:.2f}")
        print(f"  → Each point increase in location rating increases price by ${coef:.2f}k")

print("\n" + "="*70)
print("Remember: These effects are estimated while holding other variables constant!")

### 📝 TO DO #2: Interactive Coefficient Explorer

1. Run the cell below to create an interactive tool
2. Use the sliders to adjust house characteristics
3. Observe how the predicted price changes
4. Try to find:
   - The most expensive house configuration
   - The least expensive house configuration
5. Which variable has the strongest impact on price?

In [ ]:
# Interactive Price Calculator using the fitted model
from ipywidgets import interact, FloatSlider, IntSlider

def predict_house_price(size, bedrooms, age, location_rating):
    '''Interactive function to predict house price based on inputs'''
    # Create input array
    inputs = pd.DataFrame({
        'const': [1],
        'Size': [size],
        'Bedrooms': [bedrooms],
        'Age': [age],
        'Location_Rating': [location_rating]
    })

    # Predict price
    predicted_price = model.predict(inputs)[0]

    # Calculate contribution of each factor
    contributions = {
        'Base': coefficients['const'],
        'Size Effect': coefficients['Size'] * size,
        'Bedrooms Effect': coefficients['Bedrooms'] * bedrooms,
        'Age Effect': coefficients['Age'] * age,
        'Location Effect': coefficients['Location_Rating'] * location_rating
    }

    # Display results
    print("🏠 HOUSE PRICE PREDICTION")
    print("="*50)
    print(f"Size: {size:.0f} sq ft")
    print(f"Bedrooms: {bedrooms}")
    print(f"Age: {age:.1f} years")
    print(f"Location Rating: {location_rating}/10")
    print("="*50)
    print("\nPrice Breakdown:")
    for factor, value in contributions.items():
        print(f"  {factor:20} ${value:8.2f}k")
    print("  " + "-"*35)
    print(f"  {'TOTAL PREDICTED PRICE':20} ${predicted_price:8.2f}k")
    print("="*50)

    # Visualize contribution
    fig, ax = plt.subplots(figsize=(10, 6))
    factors = list(contributions.keys())
    values = list(contributions.values())
    colors = ['gray', 'green', 'blue', 'red', 'purple']

    bars = ax.bar(factors, values, color=colors, alpha=0.7, edgecolor='black')
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax.set_ylabel('Contribution to Price ($1000s)', fontsize=12)
    ax.set_title(f'Price Breakdown - Total: ${predicted_price:.2f}k', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

    # Add value labels on bars
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'${val:.1f}k', ha='center', va='bottom' if val > 0 else 'top')

    plt.tight_layout()
    plt.show()

# Create interactive widgets
interact(predict_house_price,
         size=FloatSlider(min=500, max=4000, step=100, value=2000, description='Size (sq ft)'),
         bedrooms=IntSlider(min=1, max=5, step=1, value=3, description='Bedrooms'),
         age=FloatSlider(min=0, max=50, step=1, value=10, description='Age (years)'),
         location_rating=IntSlider(min=1, max=10, step=1, value=6, description='Location'))

## Section 5: Evaluating Model Fit

### R-squared vs Adjusted R-squared

**R-squared ($R^2$)**: Proportion of variance in the response variable explained by the model
- Range: 0 to 1 (higher is better)
- Problem: Always increases when adding more predictors (even useless ones!)

**Adjusted R-squared ($R^2_{adj}$)**: Modified version that penalizes unnecessary predictors
$$R^2_{adj} = 1 - \frac{(1-R^2)(n-1)}{n-k-1}$$

Where:
- $n$ = number of observations
- $k$ = number of predictors

In [ ]:
# Model evaluation metrics
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
n_observations = len(y)
n_predictors = len(X.columns)

print("📊 MODEL FIT EVALUATION")
print("="*60)
print(f"Number of observations: {n_observations}")
print(f"Number of predictors: {n_predictors}")
print("="*60)
print(f"R-squared: {r_squared:.4f}")
print(f"  → The model explains {r_squared*100:.2f}% of the variance in Price")
print(f"\nAdjusted R-squared: {adj_r_squared:.4f}")
print(f"  → After adjusting for the number of predictors: {adj_r_squared*100:.2f}%")
print("="*60)

# Visualize actual vs predicted
y_pred = model.predict(X_with_constant)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Actual vs Predicted scatter
axes[0].scatter(y, y_pred, alpha=0.5, s=30)
axes[0].plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Price ($1000s)', fontsize=12)
axes[0].set_ylabel('Predicted Price ($1000s)', fontsize=12)
axes[0].set_title(f'Actual vs Predicted Prices (R² = {r_squared:.3f})', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Residuals plot
residuals = y - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.5, s=30)
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted Price ($1000s)', fontsize=12)
axes[1].set_ylabel('Residuals ($1000s)', fontsize=12)
axes[1].set_title('Residual Plot', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 A good model should show:")
print("  1. Points close to the diagonal line in the left plot")
print("  2. Random scatter around zero in the residual plot (no patterns)")

### 📝 TO DO #3: Comparing Models

1. Run the cells below to compare different models
2. Observe how R² and Adjusted R² change
3. Answer: Why does R² always increase but Adjusted R² might decrease?
4. Which model would you choose and why?

In [ ]:
# Compare different models with varying numbers of predictors
models_to_compare = [
    ['Size'],
    ['Size', 'Bedrooms'],
    ['Size', 'Bedrooms', 'Age'],
    ['Size', 'Bedrooms', 'Age', 'Location_Rating']
]

results = []

for predictors in models_to_compare:
    X_temp = sm.add_constant(housing_data[predictors])
    temp_model = sm.OLS(y, X_temp).fit()

    results.append({
        'Predictors': ', '.join(predictors),
        'Num_Predictors': len(predictors),
        'R²': temp_model.rsquared,
        'Adj_R²': temp_model.rsquared_adj,
        'AIC': temp_model.aic,
        'BIC': temp_model.bic
    })

comparison_df = pd.DataFrame(results)

# Display comparison table
print("📊 MODEL COMPARISON")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

# Visualize the comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# R² and Adjusted R² comparison
x_pos = np.arange(len(results))
width = 0.35

axes[0].bar(x_pos - width/2, comparison_df['R²'], width, label='R²', color='steelblue', alpha=0.7)
axes[0].bar(x_pos + width/2, comparison_df['Adj_R²'], width, label='Adjusted R²', color='orange', alpha=0.7)
axes[0].set_xlabel('Model', fontsize=12)
axes[0].set_ylabel('Value', fontsize=12)
axes[0].set_title('R² vs Adjusted R² Comparison', fontsize=14, fontweight='bold')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels([f"Model {i+1}" for i in range(len(results))], rotation=0)
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# AIC and BIC comparison
axes[1].plot(comparison_df['Num_Predictors'], comparison_df['AIC'], 'o-', label='AIC', linewidth=2, markersize=8)
axes[1].plot(comparison_df['Num_Predictors'], comparison_df['BIC'], 's-', label='BIC', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Predictors', fontsize=12)
axes[1].set_ylabel('Information Criterion', fontsize=12)
axes[1].set_title('Model Selection Criteria', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Key Observations:")
print("  • R² always increases with more predictors")
print("  • Adjusted R² may decrease if predictors don't add value")
print("  • Lower AIC/BIC values indicate better models")

## Section 6: Testing for Significance

We need to test significance at two levels:
1. **Overall Model Significance** (F-test)
2. **Individual Predictor Significance** (t-tests)

In [ ]:
# Extract F-statistic and p-value
f_statistic = model.fvalue
f_pvalue = model.f_pvalue
df_model = model.df_model  # degrees of freedom for model
df_resid = model.df_resid  # degrees of freedom for residuals

print("🔬 F-TEST FOR OVERALL MODEL SIGNIFICANCE")
print("="*60)
print("Null Hypothesis (H₀): All coefficients = 0")
print("  → The model has no predictive power")
print("Alternative Hypothesis (H₁): At least one coefficient ≠ 0")
print("  → The model has predictive power")
print("="*60)
print(f"F-statistic: {f_statistic:.2f}")
print(f"Degrees of freedom: ({df_model:.0f}, {df_resid:.0f})")
print(f"P-value: {f_pvalue:.2e}")
print("="*60)

alpha = 0.05
if f_pvalue < alpha:
    print(f"✅ CONCLUSION: P-value ({f_pvalue:.2e}) < {alpha}")
    print("   We REJECT the null hypothesis.")
    print("   The model is statistically significant!")
else:
    print(f"❌ CONCLUSION: P-value ({f_pvalue:.2e}) >= {alpha}")
    print("   We FAIL TO REJECT the null hypothesis.")
    print("   The model is NOT statistically significant.")

In [ ]:
# Individual predictor significance testing
print("🔬 T-TESTS FOR INDIVIDUAL PREDICTORS")
print("="*70)
print("H₀: βⱼ = 0 (Predictor j has no effect)")
print("H₁: βⱼ ≠ 0 (Predictor j has an effect)")
print("="*70)

# Create summary table
predictor_tests = []
alpha = 0.05

for var in X.columns:
    coef = coefficients[var]
    std_err = std_errors[var]
    t_stat = model.tvalues[var]
    p_val = model.pvalues[var]
    conf_int = model.conf_int().loc[var]

    predictor_tests.append({
        'Predictor': var,
        'Coefficient': coef,
        'Std Error': std_err,
        't-statistic': t_stat,
        'P-value': p_val,
        'Significant': '✅ Yes' if p_val < alpha else '❌ No',
        '95% CI Lower': conf_int[0],
        '95% CI Upper': conf_int[1]
    })

test_results_df = pd.DataFrame(predictor_tests)
print(test_results_df.to_string(index=False))
print("="*70)

# Visualize coefficient significance
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Coefficient plot with confidence intervals
predictors = test_results_df['Predictor']
coefs = test_results_df['Coefficient']
ci_lower = test_results_df['95% CI Lower']
ci_upper = test_results_df['95% CI Upper']
colors = ['green' if p < alpha else 'red' for p in test_results_df['P-value']]

y_pos = np.arange(len(predictors))
axes[0].barh(y_pos, coefs, color=colors, alpha=0.6)
axes[0].errorbar(coefs, y_pos, xerr=[coefs - ci_lower, ci_upper - coefs],
                 fmt='none', color='black', capsize=5)
axes[0].axvline(x=0, color='black', linestyle='--', linewidth=1)
axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(predictors)
axes[0].set_xlabel('Coefficient Value', fontsize=12)
axes[0].set_title('Coefficient Estimates with 95% Confidence Intervals', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='x')

# P-value plot
p_values = test_results_df['P-value']
bars = axes[1].bar(predictors, p_values, color=colors, alpha=0.6)
axes[1].axhline(y=alpha, color='red', linestyle='--', linewidth=2, label=f'α = {alpha}')
axes[1].set_ylabel('P-value', fontsize=12)
axes[1].set_title('P-values for Individual Predictors', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, p_val in zip(bars, p_values):
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                f'{p_val:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## Section 7: Multicollinearity

### What is Multicollinearity?
**Multicollinearity** occurs when two or more predictor variables are highly correlated with each other.

### Why is it a Problem?
1. **Unstable coefficients** - Small changes in data lead to large changes in coefficients
2. **Inflated standard errors** - Makes it harder to determine significance
3. **Wrong signs** - Coefficients might have unexpected signs
4. **F-test significant but individual t-tests not significant**

### Detection Method: Variance Inflation Factor (VIF)
$$VIF_j = \frac{1}{1 - R_j^2}$$

Where $R_j^2$ is the R-squared from regressing predictor $X_j$ on **all other predictors** (excluding the response variable Y).

**Important:** To calculate $R_j^2$ for predictor $X_j$:
1. Treat $X_j$ as the **dependent variable** (response)
2. Use all **other predictor variables** (not Y) as independent variables
3. Perform this auxiliary regression: $X_j = \alpha_0 + \alpha_1X_1 + \alpha_2X_2 + ... + \alpha_{j-1}X_{j-1} + \alpha_{j+1}X_{j+1} + ... + \varepsilon$
4. Extract the $R^2$ value from this regression - this becomes $R_j^2$

**Key Point:** $R_j^2$ measures how well the other predictors can explain variable $X_j$. A high $R_j^2$ (close to 1) means $X_j$ is highly predictable from other predictors, indicating severe multicollinearity.

### VIF Interpretation:
- **VIF = 1**: No correlation with other predictors (ideal case, $R_j^2 = 0$)
- **1 < VIF < 5**: Moderate correlation (usually acceptable)
- **VIF ≥ 5-10**: High correlation (potentially problematic)
- **VIF > 10**: Serious multicollinearity problem (coefficient variance inflated by factor of 10+)

### Example:
If $R_j^2 = 0.90$ (90% of variation in $X_j$ explained by other predictors):
$$VIF_j = \frac{1}{1-0.90} = \frac{1}{0.10} = 10$$
This means the variance of the coefficient for $X_j$ is **10 times larger** than if it were uncorrelated with other predictors.

In [ ]:
# Calculate VIF for each predictor
print("📊 VARIANCE INFLATION FACTOR (VIF) ANALYSIS")
print("="*60)

vif_data = pd.DataFrame()
vif_data['Predictor'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

# Add interpretation
def interpret_vif(vif_value):
    if vif_value == 1:
        return "✅ No correlation"
    elif vif_value < 5:
        return "✅ Acceptable"
    elif vif_value < 10:
        return "⚠️ Moderate concern"
    else:
        return "❌ High multicollinearity"

vif_data['Interpretation'] = vif_data['VIF'].apply(interpret_vif)

print(vif_data.to_string(index=False))
print("="*60)

# Visualize VIF values
fig, ax = plt.subplots(figsize=(10, 6))

colors_vif = ['green' if v < 5 else 'orange' if v < 10 else 'red' for v in vif_data['VIF']]
bars = ax.bar(vif_data['Predictor'], vif_data['VIF'], color=colors_vif, alpha=0.7, edgecolor='black')

# Add reference lines
ax.axhline(y=5, color='orange', linestyle='--', linewidth=2, label='VIF = 5 (Moderate threshold)')
ax.axhline(y=10, color='red', linestyle='--', linewidth=2, label='VIF = 10 (High threshold)')

ax.set_ylabel('VIF Value', fontsize=12)
ax.set_title('Variance Inflation Factor by Predictor', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, val in zip(bars, vif_data['VIF']):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.2f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print("\n💡 Interpretation:")
if all(vif_data['VIF'] < 5):
    print("  ✅ Good news! No serious multicollinearity detected.")
    print("  All VIF values are below 5, indicating acceptable correlation levels.")
elif any(vif_data['VIF'] >= 10):
    print("  ❌ Warning! High multicollinearity detected.")
    print("  Consider removing or combining highly correlated predictors.")
else:
    print("  ⚠️ Moderate multicollinearity detected.")
    print("  Monitor the situation but it may be acceptable.")

## Section 8: Real-World Example - California Housing Dataset

Let's apply what we've learned to a real dataset to see how multiple regression works in practice.

In [ ]:
# Load California housing dataset (simplified version)
from sklearn.datasets import fetch_california_housing

# Fetch the dataset
california = fetch_california_housing()
california_df = pd.DataFrame(california.data, columns=california.feature_names)
california_df['Price'] = california.target  # Median house value in hundreds of thousands

# Select relevant features similar to our examples
# Rename for clarity
california_subset = pd.DataFrame({
    'Price': california_df['Price'] * 100,  # Convert to thousands
    'Income': california_df['MedInc'] * 10,  # Median income (in $1000s)
    'House_Age': california_df['HouseAge'],  # Average age
    'Rooms': california_df['AveRooms'],  # Average rooms per household
    'Population': california_df['Population'] / 1000,  # Population in thousands
    'Latitude': california_df['Latitude']
})

# Take a sample for faster computation
california_sample = california_subset.sample(n=500, random_state=42)

print("🏡 CALIFORNIA HOUSING DATASET")
print("="*60)
print(f"Sample size: {len(california_sample)} houses")
print("\nDataset Overview:")
print(california_sample.describe().round(2))

# Fit multiple regression model
X_cal = california_sample[['Income', 'House_Age', 'Rooms', 'Population']]
y_cal = california_sample['Price']
X_cal_const = sm.add_constant(X_cal)
model_cal = sm.OLS(y_cal, X_cal_const).fit()

print("\n" + "="*60)
print("REGRESSION RESULTS - CALIFORNIA HOUSING")
print("="*60)
print(f"R-squared: {model_cal.rsquared:.4f}")
print(f"Adjusted R-squared: {model_cal.rsquared_adj:.4f}")
print("\nCoefficients and Significance:")
print("-"*60)

for var in X_cal.columns:
    coef = model_cal.params[var]
    pval = model_cal.pvalues[var]
    sig = "✅ Significant" if pval < 0.05 else "❌ Not Significant"

    if var == 'Income':
        print(f"{var:15} {coef:10.2f} (p={pval:.3f}) {sig}")
        print(f"{'':15} → Each $1k increase in median income → ${coef:.2f}k price increase")
    elif var == 'House_Age':
        print(f"\n{var:15} {coef:10.2f} (p={pval:.3f}) {sig}")
        print(f"{'':15} → Each year older → ${abs(coef):.2f}k price change")
    elif var == 'Rooms':
        print(f"\n{var:15} {coef:10.2f} (p={pval:.3f}) {sig}")
        print(f"{'':15} → Each additional room → ${coef:.2f}k price increase")
    else:  # Population
        print(f"\n{var:15} {coef:10.2f} (p={pval:.3f}) {sig}")
        print(f"{'':15} → Each 1k population increase → ${coef:.2f}k price change")

print("="*60)

# Create actual vs predicted plot for real data
y_pred_cal = model_cal.predict(X_cal_const)

plt.figure(figsize=(10, 6))
plt.scatter(y_cal, y_pred_cal, alpha=0.5, s=20)
plt.plot([y_cal.min(), y_cal.max()], [y_cal.min(), y_cal.max()], 'r--', lw=2)
plt.xlabel('Actual Price ($1000s)', fontsize=12)
plt.ylabel('Predicted Price ($1000s)', fontsize=12)
plt.title(f'California Housing: Actual vs Predicted Prices (R² = {model_cal.rsquared:.3f})',
          fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 Real-World Insights:")
print("  • Income is typically the strongest predictor of house prices")
print("  • The model explains about {:.1f}% of price variation".format(model_cal.rsquared * 100))
print("  • Real data is messier than simulated data!")

## Summary: Key Takeaways from Multiple Regression

### 📚 What We've Learned:

1. **Multiple Regression Model**
   - Extends simple regression to multiple predictors
   - Equation: $Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + ... + \beta_k X_k + \epsilon$
   - Fits a hyperplane in higher dimensions

2. **Coefficient Interpretation**
   - Each $\beta_i$ represents change in Y for one-unit change in $X_i$
   - **Key:** "Holding all other variables constant"
   - Different from simple regression due to controlling for other variables

3. **Model Evaluation**
   - **R-squared:** Proportion of variance explained
   - **Adjusted R-squared:** Penalizes unnecessary predictors
   - Use Adjusted R-squared for model comparison

4. **Significance Testing**
   - **F-test:** Tests overall model significance
   - **t-tests:** Test individual predictor significance
   - Both tests are important for model validation

5. **Multicollinearity**
   - Problem when predictors are highly correlated
   - Detected using VIF (Variance Inflation Factor)
   - VIF > 5-10 indicates potential problems
   - Solution: Remove or combine correlated predictors

### 🎯 Practical Tips:
- Always check for multicollinearity before interpreting coefficients
- Use Adjusted R-squared when comparing models
- Consider both statistical and practical significance
- Visualize your data and residuals to check assumptions
- Real-world data is messier than textbook examples!

### 🚀 Next Steps:
- Practice with different datasets
- Learn about model selection techniques
- Explore regularization methods (Ridge, Lasso)
- Study logistic regression for classification problems

In [ ]:
# Final Interactive Summary Tool
print("🎓 MULTIPLE REGRESSION QUICK REFERENCE")
print("="*70)

reference = {
    'Concept': [
        'Model Equation',
        'Coefficient Interpretation',
        'R-squared',
        'Adjusted R-squared',
        'F-test',
        't-test',
        'VIF < 5',
        'VIF 5-10',
        'VIF > 10'
    ],
    'Description': [
        'Y = β₀ + β₁X₁ + β₂X₂ + ... + βₖXₖ + ε',
        'Change in Y per unit change in Xᵢ, others constant',
        'Proportion of variance explained (0-1)',
        'R² adjusted for number of predictors',
        'Tests if model is significant overall',
        'Tests if individual predictor is significant',
        'No multicollinearity concern',
        'Moderate multicollinearity',
        'Severe multicollinearity problem'
    ],
    'Key Point': [
        'Extends to multiple predictors',
        'Control for other variables',
        'Always increases with more predictors',
        'Can decrease if predictor unhelpful',
        'H₀: All βᵢ = 0',
        'H₀: βᵢ = 0',
        '✅ Good',
        '⚠️ Monitor',
        '❌ Remove predictors'
    ]
}

reference_df = pd.DataFrame(reference)
print(reference_df.to_string(index=False))
print("="*70)

print("\n📝 Remember:")
print("  • Multiple regression is powerful but requires careful interpretation")
print("  • Always check assumptions and diagnostics")
print("  • Practice makes perfect!")
print("\n🎉 Congratulations on completing Module 7: Multiple Regression!")